# Asistente de Documentos con Gemini y Gradio

En los notebooks anteriores construimos chatbots que responden preguntas generales.
En este notebook damos un paso más: construir un asistente que responda preguntas
sobre un documento específico — el paper seminal de los Transformers:
**"Attention is All You Need"** (Vaswani et al., 2017).

La técnica que usamos se llama **inyección de contexto completo**:
extraemos el texto del PDF y lo embebemos directamente en el `system_instruction`
de cada llamada a Gemini. Esto funciona porque:

- El paper tiene ~22 páginas (~40K tokens)
- `gemini-2.5-flash-lite` tiene una ventana de contexto de **1,000,000 tokens**
- Es la base conceptual de **RAG**, uno de los patrones más usados en IA en producción

## Arquitectura

```
PDF  →  pypdf (extracción)  →  system_prompt  →  Gemini API  →  Gradio UI
```

## Prerrequisitos

- Archivo `.env` con `GEMINI_API_KEY` en la raíz del workspace
- `AIAYN.pdf` en la raíz del workspace
- Dependencias instaladas: `pypdf`, `gradio`, `google-genai`, `python-dotenv`

In [11]:
# %%bash
# pip install pypdf gradio google-genai python-dotenv

## 1. Extracción de Texto del PDF

Lo primero es leer el PDF y convertirlo a texto plano.
`pypdf` hace esto en pocas líneas — iteramos página por página y
concatenamos el texto extraído.

Esta es la pieza clave: sin el texto del documento, el modelo no tiene
contexto para responder preguntas específicas sobre él.

In [12]:
from pathlib import Path
from pypdf import PdfReader


def find_project_root() -> Path:
    """Encuentra la raíz del workspace buscando AIAYN.pdf desde el directorio actual."""
    # VS Code puede ejecutar el notebook desde la raíz del workspace o desde work_resources/
    # Probamos ambas ubicaciones para que funcione en cualquier caso
    for candidate in [Path.cwd(), Path.cwd().parent]:
        if (candidate / "AIAYN.pdf").exists():
            return candidate
    raise FileNotFoundError("No se encontró AIAYN.pdf. Verifica que el PDF esté en la raíz del workspace.")


def extract_text_from_pdf(pdf_path: Path) -> str:
    """Extrae el texto completo de un PDF página por página."""
    reader = PdfReader(str(pdf_path))
    partes = []

    for page in reader.pages:
        # extract_text() puede retornar None en páginas vacías o con solo imágenes
        texto = page.extract_text() or ""
        partes.append(texto)

    # Unimos las páginas con salto de línea para preservar la estructura
    return "\n".join(partes)


# Localizamos el proyecto y el PDF de forma robusta (funciona con cualquier CWD)
PROJECT_ROOT = find_project_root()
PDF_PATH = PROJECT_ROOT / "AIAYN.pdf"

print(f"Raíz del proyecto: {PROJECT_ROOT}")
print(f"PDF encontrado en: {PDF_PATH}")

document_text = extract_text_from_pdf(PDF_PATH)

# Verificamos que la extracción funcionó correctamente
reader_info = PdfReader(str(PDF_PATH))
print(f"\nPáginas en el PDF: {len(reader_info.pages)}")
print(f"Caracteres extraídos: {len(document_text):,}")
print("\n--- Primeros 500 caracteres ---")
print(document_text[:500])

Raíz del proyecto: d:\SEM-2026-1\IA\WORKSHOP3CLAUDIO\intro_IA_curso2026\workshop3
PDF encontrado en: d:\SEM-2026-1\IA\WORKSHOP3CLAUDIO\intro_IA_curso2026\workshop3\AIAYN.pdf

Páginas en el PDF: 15
Caracteres extraídos: 39,629

--- Primeros 500 caracteres ---
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗ †
University of Toronto
aidan@cs.toronto.edu
Łukasz 


## 2. Configuración del Entorno

Igual que en los notebooks anteriores: cargamos la API key desde `.env`,
inicializamos el cliente de Gemini y definimos el modelo a usar.

Usamos `gemini-2.5-flash-lite` porque su ventana de 1M tokens puede
contener cómodamente el paper completo en el system prompt.

In [13]:
import os

import gradio as gr
from dotenv import load_dotenv
from google import genai
from google.genai import types

# load_dotenv() sin argumentos busca .env desde el CWD hacia arriba automáticamente
# Esto funciona independientemente de si VS Code ejecuta desde la raíz o desde work_resources/
load_dotenv(PROJECT_ROOT / ".env")

# Verificamos que la key esté disponible antes de continuar
if os.getenv("GEMINI_API_KEY"):
    print("Gemini API Key cargada correctamente")
else:
    print("ERROR: GEMINI_API_KEY no encontrada. Verifica tu archivo .env")

Gemini API Key cargada correctamente


In [14]:
# Inicializamos el cliente de Gemini con nuestra API key
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

# Modelo que usaremos en todo el notebook
# gemini-2.5-flash-lite: rápido, gratuito, 1M tokens de contexto
MODELO = "gemini-2.5-flash-lite"

print(f"Cliente Gemini inicializado con modelo: {MODELO}")

Cliente Gemini inicializado con modelo: gemini-2.5-flash-lite


## 3. Función de Chat con Contexto del Documento

Esta es la parte central del ejercicio. La idea es construir un `system_prompt`
que incluya el texto **completo** del paper, instruyendo al modelo a responder
**solo** basándose en ese contenido.

Usamos dos funciones:

1. **`build_system_prompt`** — construye el prompt del sistema embebiendo el documento
   dentro de etiquetas `<documento>...</documento>` para delimitar claramente el contexto.

2. **`chat_con_documento`** — maneja el historial y genera respuestas en streaming
   usando el mismo patrón `yield` que vimos en el notebook 04.

### ¿Por qué etiquetas XML en el prompt?

Las etiquetas `<documento>` ayudan al modelo a distinguir entre las instrucciones
del sistema y el contenido del documento, reduciendo la posibilidad de confusión
entre ambos contextos.

In [15]:
def build_system_prompt(document_text: str) -> str:
    """Construye el system prompt embebiendo el documento completo como contexto."""
    return f"""Eres un asistente experto en el paper "Attention Is All You Need" (Vaswani et al., 2017).

Debes responder las preguntas del usuario basándote EXCLUSIVAMENTE en el contenido del documento
que aparece a continuación entre las marcas <documento> y </documento>.

Reglas estrictas:
1. Responde SOLO con información que esté explícitamente en el documento.
2. NO uses conocimiento externo, incluso si conoces la respuesta por otros medios.
3. Si la información no aparece en el documento, responde exactamente:
   "Esa información no se encuentra en el documento proporcionado."
4. Cuando sea útil, cita textualmente fragmentos cortos del documento para respaldar tu respuesta.
5. Responde en el idioma en que te pregunten (por defecto español).

<documento>
{document_text}
</documento>
"""

In [16]:
def _get_text(content) -> str:
    """Extrae el texto sin importar si llega como string o lista."""
    if isinstance(content, str):
        return content
    if isinstance(content, list) and content:
        first = content[0]
        if isinstance(first, dict):
            return first.get("text", "")
        return str(first)
    return str(content)


def chat_con_documento(message: str, history: list, document_text: str):
    """
    Función de chat que responde sobre el documento usando streaming.

    Parámetros que recibe de Gradio:
    - message: el mensaje actual del usuario
    - history: lista de tuplas [user_msg, assistant_msg] de turnos anteriores
    - document_text: el texto extraído del PDF (pasado desde additional_inputs)
    """

    # Construimos el system prompt con el documento embebido
    system_prompt = build_system_prompt(document_text)

    # Convertimos el historial de Gradio (tuplas) al formato que espera Gemini
    # En Gemini el rol del asistente es "model", no "assistant"
    contenido = []
    for user_msg, assistant_msg in history:
        contenido.append(
            types.Content(role="user", parts=[types.Part(text=_get_text(user_msg))])
        )
        if assistant_msg:
            contenido.append(
                types.Content(role="model", parts=[types.Part(text=_get_text(assistant_msg))])
            )

    # Agregamos el mensaje actual del usuario al final del historial
    contenido.append(
        types.Content(role="user", parts=[types.Part(text=message)])
    )

    # Streaming: acumulamos la respuesta y la devolvemos parcialmente con yield
    # Gradio detecta el yield automáticamente y muestra el texto en tiempo real
    respuesta = ""
    for chunk in client.models.generate_content_stream(
        model=MODELO,
        config=types.GenerateContentConfig(system_instruction=system_prompt),
        contents=contenido,
    ):
        if chunk.text:
            respuesta += chunk.text
            yield respuesta


## 4. Interfaz Gradio

Usamos `gr.ChatInterface` — el mismo componente de alto nivel que vimos en el
notebook 04 — porque maneja automáticamente la visualización del historial,
burbujas de mensajes y el campo de entrada.

El truco para pasar el texto del documento a la función es `additional_inputs`
con un `gr.Textbox(visible=False)`: Gradio lo pasa automáticamente como tercer
argumento a `chat_con_documento` sin mostrarlo en la interfaz.

El historial llega como lista de tuplas `[user_msg, assistant_msg]`, que es el
formato estándar de `gr.ChatInterface` — exactamente el que maneja nuestra función.


In [17]:
# Informamos cuánto contexto estamos inyectando
print(f"Páginas del paper: {len(PdfReader(str(PDF_PATH)).pages)}")
print(f"Caracteres del documento: {len(document_text):,}")
# Estimación aproximada: ~4 caracteres por token en inglés técnico
tokens_estimados = len(document_text) // 4
print(f"Tokens estimados en system prompt: ~{tokens_estimados:,}")
print(f"Porcentaje del límite de contexto (1M tokens): {tokens_estimados/1_000_000*100:.1f}%")

Páginas del paper: 15
Caracteres del documento: 39,629
Tokens estimados en system prompt: ~9,907
Porcentaje del límite de contexto (1M tokens): 1.0%


In [18]:
# Construimos la interfaz del asistente de documentos
demo = gr.ChatInterface(
    fn=chat_con_documento,
    title="Asistente del Paper: Attention Is All You Need",
    description=(
        "Haz preguntas sobre el paper de Vaswani et al. (2017). "
        "El asistente responde **exclusivamente** con información contenida en el documento."
    ),
    additional_inputs=[
        # Pasamos el texto del documento sin mostrarlo en la UI
        gr.Textbox(value=document_text, visible=False)
    ],
    # Con additional_inputs, cada ejemplo debe ser una lista: [mensaje, valor_adicional]
    # El segundo elemento corresponde al Textbox oculto con el texto del documento
    examples=[
        ["¿Cuál es la arquitectura principal propuesta en el paper?", document_text],
        ["¿Qué es el mecanismo de atención?", document_text],
        ["¿Cuántas capas tiene el encoder del modelo base?", document_text],
        ["¿Quiénes son los autores del paper?", document_text],
        ["¿Cuál es el resultado del modelo en la tarea WMT 2014 English-to-German?", document_text],
    ],
    flagging_mode="never",
)


In [19]:
# Lanzamos la interfaz — disponible en http://127.0.0.1:8080
demo.launch(
    server_name="127.0.0.1",
    server_port=8080,
    show_error=True
)


* Running on local URL:  http://127.0.0.1:8080
* To create a public link, set `share=True` in `launch()`.


In [20]:
# Ejecutar esta celda para cerrar el servidor Gradio
demo.close()

Closing server running on port: 8080
